In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import duckdb
import logging
import requests
import geopandas as gpd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

In [3]:
# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [4]:
# Load Fire and Resource Assessment Program (FRAP) dataset 
frap = pd.read_csv('https://gis.data.cnra.ca.gov/api/download/v1/items/c3c10388e3b24cec8a954ba10458039d/csv?layers=0')

# Select relevant columns for fire table / drop irrelevant
fires_df = frap.drop(columns=['Agency', 'Complex ID', 'DECADES', 'Comments', 'Fire Number (historical use)', 'Complex Name', 
                              'Management Objective', 'Local Incident Number', 'IRWIN ID', 'Year', 'Fire Name'])

# Rename columns for consistency
fires_df.rename(columns={
    'OBJECTID': 'fire_id',
    'State': 'state',
    'Alarm Date': 'alarm_date',
    'Containment Date': 'containment_date',
    'Cause': 'cause_id',
    'Collection Method': 'collection_method',
    'GIS Calculated Acres': 'acres_burned',
    'Shape__Area': 'shape_area',
    'Shape__Length': 'shape_length',
    'Unit ID': 'unit_id_frap'}, 
    inplace=True)

# Filter the dataset to include only fires that occurred in California
fires_df = fires_df[fires_df['state'] == 'CA']

# Convert dates to datetime
fires_df['alarm_date'] = pd.to_datetime(fires_df['alarm_date'], errors='coerce')
fires_df['containment_date'] = pd.to_datetime(fires_df['containment_date'], errors='coerce')

# Filter the dataset to include only fires that occurred between 1980 and 2025
fires_df = fires_df[(fires_df['alarm_date'].dt.year >= 1980) & (fires_df['alarm_date'].dt.year <= 2025)]

# Change collection_method to a categorical variable
collection_method_mapping = {1: 'GPS Ground', 2: 'GPS Air', 3: 'Infrared', 4: 'Other Imagery',
                             5: 'Photo Interpretation', 6: 'Hand Drawn', 7: 'Mixed Collection Methods', 8: 'Unknown'}
fires_df['collection_method_numerical'] = fires_df['collection_method']
fires_df['collection_method'] = fires_df['collection_method_numerical'].map(collection_method_mapping).astype('category')

# Compute duration_days representing the number of days between alarm and containment
fires_df['duration_days'] = (fires_df['containment_date'] - fires_df['alarm_date']).dt.days

logging.info(f"Dataset processed successfully. Number of records: {len(fires_df)}")

fires_df.head()

2026-03-27 22:59:10,968 - INFO - Dataset processed successfully. Number of records: 11932


,fire_id,state,unit_id_frap,alarm_date,containment_date,cause_id,collection_method,acres_burned,shape_area,shape_length,collection_method_numerical,duration_days
0,1,CA,LDF,2025-01-07 08:00:00,2025-01-31 08:00:00,14,Mixed Collection Methods,23448.8800,1.386518e+08,140231.608232,7.0,24.0
1,2,CA,LAC,2025-01-08 08:00:00,2025-01-31 08:00:00,14,Mixed Collection Methods,14056.2600,8.336393e+07,104933.207224,7.0,23.0
2,3,CA,ANF,2025-01-22 08:00:00,2025-01-28 08:00:00,14,Mixed Collection Methods,10396.8000,6.216064e+07,96698.599858,7.0,6.0
3,4,CA,VNC,2025-01-09 08:00:00,2025-02-04 08:00:00,14,GPS Air,998.7378,5.919678e+06,15602.004849,2.0,26.0
4,5,CA,LDF,2025-01-07 08:00:00,2025-01-09 08:00:00,14,Mixed Collection Methods,831.3855,4.946082e+06,16094.217073,7.0,2.0


In [5]:
# Begin relational database design by creating separate tables for Causes, location specific weather, and Time to normalize the data

# Create Causes table
# Get unique cause_id values and create a new DataFrame for causes
causes_df = fires_df[['cause_id']].drop_duplicates().reset_index(drop=True)

# Mapping for descriptive names (directly from FRAP documentation: https://34c031f8-c9fd-4018-8c5a-4159cdff6b0d-cdn-endpoint.azureedge.net/-/media/calfire-website/what-we-do/fire-resource-assessment-program---frap/data-dictionary-updated-april-2025.pdf?rev=99e4dccd9d654a6f9ed9556c0b7e3445&hash=371C2A47D6A96261C91E0008A1BABC20)
cause_mapping = {1: "Lightning",
                 2: "Equipment Use",
                 3: "Smoking",
                 4: "Campfire",
                 5: "Debris",
                 6: "Railroad",
                 7: "Arson",
                 8: "Playing with fire",
                 9: "Miscellaneous",
                 10: "Vehicle",
                 11: "Powerline",
                 12: "Firefighter Training",
                 13: "Non-Firefighter Training",
                 14: "Unknown/Unidentified",
                 15: "Structure",
                 16: "Aircraft",
                 17: "Volcanic",
                 18: "Escaped Prescribed Burn",
                 19: "Illegal Alien Campfire"}

# Map cause_id to cause_name
causes_df['cause_name'] = causes_df['cause_id'].map(cause_mapping)

# Make cause_type column with natural or human cause types
def classify_cause(cause_id):
    if cause_id in [1, 17]:  # Natural causes
        return 'Natural'
    elif cause_id in [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 18, 19]:  # Human causes
        return 'Human'
    else:
        return 'Unknown'
    
causes_df['cause_type'] = causes_df['cause_id'].apply(classify_cause)

# Sort causes by cause_id
causes_df.sort_values('cause_id', inplace=True)
causes_df.head()

# Merge the normalized table back into the main fires_df to maintain relational structure
fires_df = fires_df.merge(causes_df, on='cause_id', how='left')
logging.info("Causes table created and merged successfully.")

2026-03-27 22:59:11,025 - INFO - Causes table created and merged successfully.


In [6]:
# Create Weather/Location table

# Load in CALFIRE Administrative Units dataset to pull out relative location information for fires
units_gdf = gpd.read_file("https://gis.data.cnra.ca.gov/api/download/v1/items/b89a2c4bbe574091b418ac83299a5f2c/shapefile?layers=0")

# Goal: Get lat/lon coordinates for each fire based on the centroid of the administrative unit it occurred in.
# Convert to a projection that is better for spatial calculations
units_proj = units_gdf.to_crs(epsg=3857)

# Compute centroid
units_proj['centroid'] = units_proj.geometry.centroid

# Convert centroid back to lat/lon
centroids_wgs84 = units_proj['centroid'].to_crs(epsg=4326)

# Assign back to original dataframe
units_gdf['lat'] = centroids_wgs84.y
units_gdf['lon'] = centroids_wgs84.x

# Rename UNITCODE to unit_id  and UNIT as unit_name for merging
units_gdf.rename(columns={'UNITCODE': 'unit_id', 'UNIT': 'unit_name'}, inplace=True)
location_df = units_gdf[['unit_id', 'unit_name', 'lat', 'lon']]

# Merge the normalized table back into the main fires_df to maintain relational structure
fires_df = fires_df.merge(location_df, left_on='unit_id_frap', right_on='unit_id', how='left')
fires_df = fires_df.dropna(subset=['lat', 'lon']).reset_index(drop=True)
logging.info("Weather/Location table created and merged successfully.")

2026-03-27 22:59:13,625 - INFO - Weather/Location table created and merged successfully.


In [7]:
# Only keep the units that are in the Administrative Units dataset (these units will have the lat/lon coordinates we need for weather data)
fires_df = fires_df[~fires_df['lat'].isna()].reset_index(drop=True)

In [8]:
# Time table
# Extract unique alarm dates
time_df = fires_df[['alarm_date']].drop_duplicates().reset_index(drop=True)

# Extract year, month, decade 
time_df['year'] = time_df['alarm_date'].dt.year
time_df['month'] = time_df['alarm_date'].dt.month
time_df['decade'] = (time_df['year'] // 10) * 10
time_df['day_of_year'] = time_df['alarm_date'].dt.dayofyear
time_df['week'] = time_df['alarm_date'].dt.isocalendar().week
time_df['is_peak_fire_season'] = time_df['month'].isin([6,7,8,9]).astype(int)

# Define seasons using a function
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:  # 9, 10, 11
        return 'Fall'

time_df['season'] = time_df['month'].apply(get_season)

# Add a primary key for relational mapping
time_df['time_id'] = time_df.index + 1

# Reorder columns
time_df = time_df[['time_id', 'alarm_date', 'year', 'month', 'decade', 'day_of_year', 'week', 'season', 'is_peak_fire_season']]

# Merge the normalized table back into the main fires_df to maintain relational structure
fires_df = fires_df.merge(time_df[['alarm_date', 'time_id']], on='alarm_date', how='left')
logging.info("Time table created and merged successfully.")

2026-03-27 22:59:13,650 - INFO - Time table created and merged successfully.


In [9]:
# Weather Fetch Function to pull historical weather data for each fire based on its lat/lon and alarm date using the Open-Meteo API
def fetch_weather(lat, lon, date, retries=3):
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": date,
        "end_date": date,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max", # Key weather variables
        "timezone": "America/Los_Angeles"}

    # Try fetching weather data with retries in case of any issues
    for _ in range(retries):
        try:
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()

            data = response.json()["daily"]

            return {"temp_max": data["temperature_2m_max"][0],
                    "temp_min": data["temperature_2m_min"][0],
                    "precip": data["precipitation_sum"][0],
                    "wind_speed": data["windspeed_10m_max"][0]}
        except:
            time.sleep(1)

    return None

# Parallel Weather Fetching to speed up the process of fetching weather data for each unique fire 
def process_row(row):
    date_str = row['alarm_date'].strftime('%Y-%m-%d')
    weather = fetch_weather(row['lat'], row['lon'], date_str) # Fetch weather data for the fire's location and alarm date

    # If weather data couldn't be fetched, return None values for weather variables
    if weather is None:
        weather = {"temp_max": None,
                   "temp_min": None,
                   "precip": None,
                   "wind_speed": None}
    # Return a dictionary with lat, lon, alarm_date, and the fetched weather data for merging back into the main fires_df
    return {'lat': row['lat'],
            'lon': row['lon'],
            'alarm_date': row['alarm_date'],
            **weather}

# Get unique combinations of lat, lon, and alarm_date to fetch weather data for each fire without redundant API calls
weather_keys = fires_df[['lat', 'lon', 'alarm_date']].drop_duplicates().reset_index(drop=True)
weather_records = []
total = len(weather_keys)
start_time = time.time()

MAX_WORKERS = 8 

# Use ThreadPoolExecutor to fetch weather data in parallel for each unique fire location and date, while tracking progress and estimating remaining time
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    # Submit a separate task for each row in weather_keys to fetch weather data in parallel
    futures = [executor.submit(process_row, row) for _, row in weather_keys.iterrows()]

    for i, future in enumerate(as_completed(futures)):
        weather_records.append(future.result())

        # Track progress
        if i % 100 == 0 and i > 0:
            elapsed = time.time() - start_time
            rate = i / elapsed
            remaining = (total - i) / rate
            print(f"Progress: {i}/{total} | {i/total:.1%} | ETA: {remaining/60:.1f} min")
            logging.info(f"Weather Progress: {i}/{total} | {i/total:.1%} | ETA: {remaining/60:.1f} min")

# Create a DataFrame from the weather records to merge back into the main fires_df
weather_df = pd.DataFrame(weather_records)
fires_df = fires_df.merge(weather_df, on=['lat', 'lon', 'alarm_date'], how='left')
logging.info("Weather data fetched and merged successfully.")

2026-03-27 23:00:00,713 - INFO - Weather Progress: 100/5709 | 1.8% | ETA: 44.0 min


Progress: 100/5709 | 1.8% | ETA: 44.0 min


2026-03-27 23:00:08,088 - INFO - Weather Progress: 200/5709 | 3.5% | ETA: 25.0 min


Progress: 200/5709 | 3.5% | ETA: 25.0 min


2026-03-27 23:00:14,701 - INFO - Weather Progress: 300/5709 | 5.3% | ETA: 18.3 min


...

2026-03-27 23:09:59,580 - INFO - Weather Progress: 5600/5709 | 98.1% | ETA: 0.2 min


Progress: 5600/5709 | 98.1% | ETA: 0.2 min


2026-03-27 23:10:52,542 - INFO - Weather Progress: 5700/5709 | 99.8% | ETA: 0.0 min


Progress: 5700/5709 | 99.8% | ETA: 0.0 min


2026-03-27 23:10:56,978 - INFO - Weather data fetched and merged successfully.


In [10]:
# Add a primary key
weather_df['weather_id'] = weather_df.index + 1
fires_df = fires_df.merge(weather_df[['lat', 'lon', 'alarm_date', 'weather_id']], on=['lat', 'lon', 'alarm_date'], how='left')
weather_df.drop(columns=['alarm_date'], inplace=True)

In [11]:
weather_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5709 entries, 0 to 5708
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   lat         5709 non-null   float64
 1   lon         5709 non-null   float64
 2   temp_max    5022 non-null   float64
 3   temp_min    5022 non-null   float64
 4   precip      5022 non-null   float64
 5   wind_speed  5022 non-null   float64
 6   weather_id  5709 non-null   int64  
dtypes: float64(6), int64(1)
memory usage: 312.3 KB


In [12]:
# Drop columns from fires_df that are now represented in the normalized tables to avoid redundancy and maintain a clean relational structure
fires_df = fires_df.drop(columns=[
    'cause_name',
    'cause_type',
    'containment_date',
    'collection_method_numerical',
    'state',
    'alarm_date',
    'unit_name',
    'unit_id_frap',
    'lat',
    'lon',
    'temp_max',
    'temp_min',
    'precip',
    'wind_speed'])

# Reorder columns 
fires_df = fires_df[['fire_id', 'cause_id', 'unit_id', 'time_id', 'weather_id', 'collection_method', 'acres_burned', 'shape_area', 'shape_length', 'duration_days']]
logging.info("Final fires_df cleaned and structured successfully.")

2026-03-27 23:10:57,020 - INFO - Final fires_df cleaned and structured successfully.


In [13]:
# Connect and create tables

def create_database():
    try:
        # Connect to local DuckDB instance
        con = duckdb.connect(database='california_fires.duckdb', read_only=False)
        logging.info("Connected to DuckDB database 'california_fires.duckdb'")
        print("Connected to DuckDB database 'california_fires.duckdb'")

        # Drop tables if they already exist
        con.execute("DROP TABLE IF EXISTS Fires")
        con.execute("DROP TABLE IF EXISTS Causes")
        con.execute("DROP TABLE IF EXISTS Time")
        con.execute("DROP TABLE IF EXISTS Weather")
        logging.info("Existing tables dropped if they existed.")

        # Create tables
        con.execute("CREATE TABLE Causes AS SELECT * FROM causes_df")
        con.execute("CREATE TABLE Weather AS SELECT * FROM weather_df")
        con.execute("CREATE TABLE Time AS SELECT * FROM time_df")
        con.execute("CREATE TABLE Fires AS SELECT * FROM fires_df")
        logging.info("Tables created successfully in DuckDB database.")
        print("Tables created successfully in DuckDB database.")

    except Exception as e:
        logging.error(f"An error occurred: {e}")
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    create_database()

2026-03-27 23:10:57,167 - INFO - Connected to DuckDB database 'california_fires.duckdb'
2026-03-27 23:10:57,177 - INFO - Existing tables dropped if they existed.


Connected to DuckDB database 'california_fires.duckdb'


2026-03-27 23:10:57,218 - INFO - Tables created successfully in DuckDB database.


Tables created successfully in DuckDB database.


In [14]:
# Save each table as parquet file
fires_df.to_parquet('./parquet_files/fires.parquet', index=False)
causes_df.to_parquet('./parquet_files/causes.parquet', index=False)
weather_df.to_parquet('./parquet_files/weather.parquet', index=False)
time_df.to_parquet('./parquet_files/time.parquet', index=False)

logging.info("All tables saved as parquet files.")
print("All tables saved as parquet files.")

2026-03-27 23:10:57,395 - INFO - All tables saved as parquet files.


All tables saved as parquet files.
